In [1]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI


In [18]:
import os


model = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",   # or "gemini-1.5-pro", "gemini-1.5-flash"
    google_api_key=os.getenv("GOOGLE_API_KEY"),  # or set GOOGLE_API_KEY env var
)


In [35]:
# agent = create_agent(model, tools=[word_count], checkpointer=InMemorySaver())



# bishal = {"configurable": {"thread_id": "student-bishal"}}

# def say(config, text):
#     r = agent.invoke({"messages": [{"role": "user", "content": text}]}, config)
#     print("agent:", r["messages"][-1].content)

# say(bishal, "My name is Bishal.")     # stored on this thread
# say(bishal, "What is my name?")       # -> recalled from memory



In [5]:
me = {"configurable": {"thread_id": "another_thread_test"}} 

In [9]:


import sqlite3

DB_PATH = "notes.db"

def init_db():
    conn = sqlite3.connect(DB_PATH)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS notes (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            text TEXT NOT NULL,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    """)
    conn.commit()
    conn.close()

In [28]:



@tool
def save_note(text: str) -> str:
    """Save a fact the user wants remembered for later into the database i.e, sqlite"""
    conn = sqlite3.connect(DB_PATH)
    conn.execute("INSERT INTO notes (text) VALUES (?)", (text,))
    conn.commit()
    conn.close()
    return "saved"


init_db()

@tool
def list_notes() -> str:
    """List everything the user has asked to remember from the database"""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.execute(
        "SELECT text FROM notes ORDER BY id ASC"
    )
    notes = [row[0] for row in cursor.fetchall()]
    conn.close()

    return "; ".join(notes) or "(nothing saved yet)"

@tool
def delete_note(note_id: int) -> str:
    """Delete a specific saved note by its ID."""
    conn = sqlite3.connect(DB_PATH)

    cursor = conn.execute(
        "DELETE FROM notes WHERE id = ?",
        (note_id,)
    )

    conn.commit()
    deleted = cursor.rowcount
    conn.close()

    if deleted:
        return f"note {note_id} deleted"
    return f"note {note_id} not found"


In [37]:

TOOLS = [ save_note, list_notes, delete_note]
assistant = create_agent(model, tools=TOOLS, checkpointer=InMemorySaver())
me = {"configurable": {"thread_id": "another_thread_test"}} 

def say(config, text):
    r = assistant.invoke({"messages": [{"role": "user", "content": text}]}, config)
    print("agent:", r["messages"][-1].content)

    
#say(me, "save this fact 'ram is a good boy'")
say(me, "delete the note that has id 6 ")



agent: [{'type': 'text', 'text': 'I have deleted the note with ID 6.', 'extras': {'signature': 'EmgKZgERTTIPamPsnLNOsR8B1thaI8EkCfXNpbobINF6OKrTDp+UoU2TqTRl/phAxsz60rCYli0VobNm8yGSy2WsiZDDdw/POSmIYXMOfjtYZ4e1d/hOY5MTDLpf3otWByLKw7tToSsrqg=='}}]


In [38]:
def list_notes() -> str:
    """List everything the user has asked to remember."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.execute(
        "SELECT text FROM notes ORDER BY id ASC"
    )
    notes = [row[0] for row in cursor.fetchall()]
    conn.close()

    return "; ".join(notes) or "(nothing saved yet)"


res = list_notes()

In [39]:
res

'ram is a good boy'

In [31]:
# def save_note(text: str) -> str:
#     """Save a fact the user wants remembered for later."""
#     conn = sqlite3.connect(DB_PATH)
#     conn.execute("INSERT INTO notes (text) VALUES (?)", (text,))
#     conn.commit()
#     conn.close()
#     return "saved"


# res = save_note("ram is a good boy")


In [22]:

def delete_all_notes() -> str:
    """Delete everything saved in the notes table."""
    conn = sqlite3.connect(DB_PATH)
    conn.execute("DELETE FROM notes")
    conn.commit()
    conn.close()

    return "all notes deleted"


delete_all_notes()

'all notes deleted'

In [35]:
def list_notes() -> str:
    """List everything the user has asked to remember."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.execute(
        "SELECT * FROM notes "
    )
    notes = [row[0] for row in cursor.fetchall()]
    conn.close()

    return (notes) or "(nothing saved yet)"


res = list_notes()

In [36]:
res

[5, 6]